# Parent Atlas GPU Enrichment Notebook

GPU experiments on extracted parent-atlas data.

**Data source**: legal-ai Postgres (canonical truth)
- 58,304 atlas_packets
- 7,530 atlas_summary_layers
- 105,404 atlas_tree_nodes

**Colab outputs**: Derived only (scores, embeddings, clusters)
**Local truth**: Postgres remains canonical


In [ ]:
# Install dependencies
!pip install -q pandas numpy transformers torch bitsandbytes accelerate duckdb
print("✓ Dependencies installed")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
# Load NDJSON files from Drive
import pandas as pd
import json
from pathlib import Path

# Assuming files are in /content/drive/MyDrive/colab-parent-atlas/
DATA_DIR = Path('/content/drive/MyDrive/colab-parent-atlas')

# Load packets
def load_ndjson(file_path, limit=None):
    """Load NDJSON file into DataFrame"""
    rows = []
    with open(file_path) as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

print("Loading data...")
packets = load_ndjson(DATA_DIR / 'atlas_packets.ndjson')
summary_layers = load_ndjson(DATA_DIR / 'atlas_summary_layers.ndjson')
packet_features = load_ndjson(DATA_DIR / 'packet_features.ndjson')

print(f"✓ Packets: {len(packets)} rows")
print(f"✓ Summary layers: {len(summary_layers)} rows")
print(f"✓ Packet features: {len(packet_features)} rows")
print(f"\nPackets schema: {packets.columns.tolist()[:5]}...")

In [ ]:
# Load Gemma4-E4B for enrichment
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading Gemma-4-E4B-it...")

model_id = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"✓ Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params (quantized)")

In [ ]:
# Example: Enrich top-N packets with Gemma4 summaries
from tqdm import tqdm
import gc

def enrich_packet_batch(packet_rows, model, tokenizer, max_rows=100):
    """Generate summaries for packet batch"""
    results = []
    
    for i, row in enumerate(packet_rows[:max_rows]):
        try:
            packet_key = row.get('packet_key', f'packet_{i}')
            source_ref = row.get('source_ref', '')
            feature_id = row.get('feature_id', '')
            
            prompt = f"""Summarize this code packet in 2-3 sentences:
Packet: {packet_key}
Source: {source_ref}
Feature: {feature_id}

Summary:"""
            
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer.encode(text, return_tensors="pt").to("cuda")
            
            with torch.no_grad():
                outputs = model.generate(
                    inputs,
                    max_new_tokens=150,
                    temperature=0.3,
                    top_p=0.95,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            summary_full = tokenizer.decode(outputs[0], skip_special_tokens=True)
            summary = summary_full.split("Summary:")[-1].strip()[:300]
            
            results.append({
                "packet_key": packet_key,
                "source_ref": source_ref,
                "feature_id": feature_id,
                "summary": summary,
                "model": "gemma-4-E4B-it"
            })
            
            if (i + 1) % 10 == 0:
                gc.collect()
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"Error on packet {i}: {e}")
    
    return results

# Test on first 10 packets
print("Enriching first 10 packets...")
packet_dicts = packets.head(10).to_dict('records')
enriched = enrich_packet_batch(packet_dicts, model, tokenizer, max_rows=10)
print(f"✓ Enriched {len(enriched)} packets")
print(f"\nSample: {enriched[0] if enriched else 'No results'}")

In [ ]:
# Save enriched results to NDJSON
import json
from datetime import datetime

output_file = 'parent-atlas-enriched.ndjson'

with open(output_file, 'w') as f:
    for result in enriched:
        result['enriched_at'] = datetime.now().isoformat()
        f.write(json.dumps(result) + '\n')

print(f"✓ Saved to {output_file}")
print(f"  Rows: {len(enriched)}")
print(f"  Size: {Path(output_file).stat().st_size / 1024:.1f} KB")

In [ ]:
# Download results
from google.colab import files

print(f"Downloading {output_file}...")
files.download(output_file)
print("✓ Download started")

## Next: Local Import

After download, import results locally:
```bash
cd sveltekit-frontend
npm run atlas:colab:import parent-atlas-enriched.ndjson
```

This merges Colab enrichment back into Postgres atlas_packets.
